In [4]:
import sys
print(sys.executable)


c:\Users\akand\OneDrive\Documents\data journey\Amdari Resources\DS Projects\MIG_Cement_t\mig-venv\Scripts\python.exe


In [5]:
import pandas as pd
print("Pandas is working")

import numpy as np

import warnings
warnings.filterwarnings("ignore")

# Load the forecasting-prep dataset
df = pd.read_csv("../data/MIG_features.csv", parse_dates=["date"])
df = df.set_index("date")

df.head()


Pandas is working


,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,expected_closing,...,roll_30,roll_90,month,quarter,day_of_week,rain_lag_1,temp_lag_1,behavior_aggressive,behavior_chaotic,behavior_conservative
date,,,,,,,,,,,,,,,,,,,,,
2022-03-31,SITE_001,CEM_III,46.10,19.44,0.00,19.44,0.00,4.55,20.20,0.00,...,29.936667,31.357333,3,1,3,5.82,13.85,True,False,False
2022-04-01,SITE_001,CEM_II,31.10,31.10,0.00,38.53,7.43,7.13,28.90,7.43,...,30.973333,31.319111,4,2,4,4.55,20.20,True,False,False
2022-04-02,SITE_001,CEM_II,0.00,0.00,7.43,44.69,52.12,11.28,21.26,52.12,...,29.862000,30.816222,4,2,5,7.13,28.90,True,False,False
2022-04-03,SITE_001,CEM_III,0.00,0.00,52.12,22.88,75.00,2.19,15.64,75.00,...,28.259000,30.386333,4,2,6,11.28,21.26,True,False,False
2022-04-04,SITE_001,CEM_II,46.06,46.06,75.00,23.72,52.66,1.19,18.86,52.66,...,28.176667,30.529667,4,2,0,2.19,15.64,True,False,False


In [6]:
# Load engineered features
df_features = pd.read_csv("../data/MIG_features.csv")


In [7]:
df_features['date'] = pd.to_datetime(df_features['date'])
df_features = df_features.sort_values(['site_id', 'date'])


Define features and target 

In [8]:
feature_cols = [
    'lag_1','lag_7','lag_14','lag_30',
    'roll_7','roll_30','roll_90',
    'month','quarter','day_of_week',
    'rain_lag_1','temp_lag_1',
    'behavior_aggressive','behavior_chaotic','behavior_conservative'
]
target_col = "consumed_tonnes"


Train/Test Split (time-series) - Time-series must be split by date — never randomly.

In [9]:
split_date = "2023-01-01"

train = df[df.index < split_date]
test = df[df.index >= split_date]

train.shape, test.shape


((8280, 31), (21930, 31))

Create per-site datasets - each site behaves differently — so each site gets its own model.

In [10]:
sites = df["site_id"].unique()

site_datasets = {}

for site in sites:
    site_train = train[train["site_id"] == site]
    site_test = test[test["site_id"] == site]

    X_train = site_train[feature_cols]
    y_train = site_train[target_col]

    X_test = site_test[feature_cols]
    y_test = site_test[target_col]

    site_datasets[site] = {
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "y_test": y_test
    }

site_datasets["SITE_001"]["X_train"].head()


,lag_1,lag_7,lag_14,lag_30,roll_7,roll_30,roll_90,month,quarter,day_of_week,rain_lag_1,temp_lag_1,behavior_aggressive,behavior_chaotic,behavior_conservative
date,,,,,,,,,,,,,,,
2022-03-31,13.38,53.87,20.99,41.12,23.180000,29.936667,31.357333,3,1,3,5.82,13.85,True,False,False
2022-04-01,19.44,15.12,36.86,0.00,25.462857,30.973333,31.319111,4,2,4,4.55,20.20,True,False,False
2022-04-02,31.10,36.70,42.66,33.34,20.220000,29.862000,30.816222,4,2,5,7.13,28.90,True,False,False
2022-04-03,0.00,23.80,0.00,48.09,16.820000,28.259000,30.386333,4,2,6,11.28,21.26,True,False,False
2022-04-04,0.00,24.57,38.81,48.53,19.890000,28.176667,30.529667,4,2,0,2.19,15.64,True,False,False


Train SARIMAX (for stable sites). SARIMAX can use external features (exogenous variables), which is perfect for: weather, planned pours, 
rolling averages, behavior

quick diagnostic for one site site_001

In [14]:
site = 'SITE_001'
df_site = df_features[df_features['site_id'] == site].copy()
df_site = df_site.sort_values('date').set_index('date').asfreq('D')
df_site = df_site.dropna(subset=sarimax_features + ['consumed_tonnes'])

print(df_site[sarimax_features + ['consumed_tonnes']].dtypes)
print(df_site[sarimax_features + ['consumed_tonnes']].head())


lag_1                    float64
lag_7                    float64
lag_14                   float64
lag_30                   float64
roll_7                   float64
roll_30                  float64
roll_90                  float64
month                      int64
quarter                    int64
day_of_week                int64
rain_lag_1               float64
temp_lag_1               float64
behavior_aggressive         bool
behavior_chaotic            bool
behavior_conservative       bool
consumed_tonnes          float64
dtype: object
            lag_1  lag_7  lag_14  lag_30     roll_7    roll_30    roll_90  \
date                                                                        
2022-03-31  13.38  53.87   20.99   41.12  23.180000  29.936667  31.357333   
2022-04-01  19.44  15.12   36.86    0.00  25.462857  30.973333  31.319111   
2022-04-02  31.10  36.70   42.66   33.34  20.220000  29.862000  30.816222   
2022-04-03   0.00  23.80    0.00   48.09  16.820000  28.259000  30.386333

In [15]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# GLOBAL SARIMAX FEATURES
# -----------------------------
sarimax_features = [
    'lag_1','lag_7','lag_14','lag_30',
    'roll_7','roll_30','roll_90',
    'month','quarter','day_of_week',
    'rain_lag_1','temp_lag_1',
    'behavior_aggressive','behavior_chaotic','behavior_conservative'
]

sites = df_features['site_id'].unique()
sarimax_results = {}

# -----------------------------
# SARIMAX LOOP
# -----------------------------
for site in sites:
    print(f"\nTraining SARIMAX for {site}...")

    df_site = df_features[df_features['site_id'] == site].copy()
    df_site = df_site.sort_values('date').set_index('date').asfreq('D')

    # Drop rows with missing values
    df_site = df_site.dropna(subset=sarimax_features + ['consumed_tonnes'])

    # FIX: Convert behaviour flags from bool → int
    behaviour_cols = [
        'behavior_aggressive',
        'behavior_chaotic',
        'behavior_conservative'
    ]
    df_site[behaviour_cols] = df_site[behaviour_cols].astype(int)

    # Force ALL features + target to numeric
    df_site[sarimax_features + ['consumed_tonnes']] = (
        df_site[sarimax_features + ['consumed_tonnes']]
        .apply(pd.to_numeric, errors='coerce')
    )

    # Drop rows that became NaN after coercion
    df_site = df_site.dropna(subset=sarimax_features + ['consumed_tonnes'])

    # Prepare exogenous + target
    X = df_site[sarimax_features].astype('float64')
    y = df_site['consumed_tonnes'].astype('float64')

    try:
        model = SARIMAX(
            y,
            exog=X,
            order=(1,1,1),
            enforce_stationarity=False,
            enforce_invertibility=False
        )

        results = model.fit(method='powell', disp=False)
        sarimax_results[site] = results

        print(f"SARIMAX completed for {site}")

    except Exception as e:
        print(f"❌ SARIMAX failed for {site}: {e}")



Training SARIMAX for SITE_001...
SARIMAX completed for SITE_001

Training SARIMAX for SITE_002...
SARIMAX completed for SITE_002

Training SARIMAX for SITE_003...
SARIMAX completed for SITE_003

Training SARIMAX for SITE_004...
SARIMAX completed for SITE_004

Training SARIMAX for SITE_005...
SARIMAX completed for SITE_005

Training SARIMAX for SITE_006...
SARIMAX completed for SITE_006

Training SARIMAX for SITE_007...
SARIMAX completed for SITE_007

Training SARIMAX for SITE_008...
SARIMAX completed for SITE_008

Training SARIMAX for SITE_009...
SARIMAX completed for SITE_009

Training SARIMAX for SITE_010...
SARIMAX completed for SITE_010

Training SARIMAX for SITE_011...
SARIMAX completed for SITE_011

Training SARIMAX for SITE_012...
SARIMAX completed for SITE_012

Training SARIMAX for SITE_013...
SARIMAX completed for SITE_013

Training SARIMAX for SITE_014...
SARIMAX completed for SITE_014

Training SARIMAX for SITE_015...
SARIMAX completed for SITE_015

Training SARIMAX for SIT

SARIMAX training function - SARIMAX learns: trend, weekly seasonality, noise, 
external drivers

Train XGBoost (for chaotic sites) - XGBoost handles: noise, spikes, non-linear patterns, behaviour effects, weather effects. Perfect for MIG cement.

XGBoost training

In [16]:
from xgboost import XGBRegressor
import pandas as pd

# ============================================================
# XGBOOST FEATURES (same as SARIMAX except no time index needed)
# ============================================================
xgb_features = [
    'lag_1','lag_7','lag_14','lag_30',
    'roll_7','roll_30','roll_90',
    'month','quarter','day_of_week',
    'rain_lag_1','temp_lag_1',
    'behavior_aggressive','behavior_chaotic','behavior_conservative'
]

xgb_results = {}

for site in df_features['site_id'].unique():
    print(f"\nTraining XGBoost for {site}...")

    df_site = df_features[df_features['site_id'] == site].copy()

    # Drop missing values
    df_site = df_site.dropna(subset=xgb_features + ['consumed_tonnes'])

    if df_site.empty:
        print(f"XGBoost skipped for {site}: no valid rows.")
        continue

    X = df_site[xgb_features]
    y = df_site['consumed_tonnes']

    model = XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='reg:squarederror'
    )

    model.fit(X, y)
    xgb_results[site] = model

    print(f"XGBoost completed for {site}")



Training XGBoost for SITE_001...
XGBoost completed for SITE_001

Training XGBoost for SITE_002...
XGBoost completed for SITE_002

Training XGBoost for SITE_003...
XGBoost completed for SITE_003

Training XGBoost for SITE_004...
XGBoost completed for SITE_004

Training XGBoost for SITE_005...
XGBoost completed for SITE_005

Training XGBoost for SITE_006...
XGBoost completed for SITE_006

Training XGBoost for SITE_007...
XGBoost completed for SITE_007

Training XGBoost for SITE_008...
XGBoost completed for SITE_008

Training XGBoost for SITE_009...
XGBoost completed for SITE_009

Training XGBoost for SITE_010...
XGBoost completed for SITE_010

Training XGBoost for SITE_011...
XGBoost completed for SITE_011

Training XGBoost for SITE_012...
XGBoost completed for SITE_012

Training XGBoost for SITE_013...
XGBoost completed for SITE_013

Training XGBoost for SITE_014...
XGBoost completed for SITE_014

Training XGBoost for SITE_015...
XGBoost completed for SITE_015

Training XGBoost for SIT

Train Random Forest (baseline model)

In [17]:
from sklearn.ensemble import RandomForestRegressor

# ============================================================
# RANDOM FOREST FEATURES (same as XGBoost)
# ============================================================
rf_features = [
    'lag_1','lag_7','lag_14','lag_30',
    'roll_7','roll_30','roll_90',
    'month','quarter','day_of_week',
    'rain_lag_1','temp_lag_1',
    'behavior_aggressive','behavior_chaotic','behavior_conservative'
]

rf_results = {}

for site in df_features['site_id'].unique():
    print(f"\nTraining Random Forest for {site}...")

    df_site = df_features[df_features['site_id'] == site].copy()

    df_site = df_site.dropna(subset=rf_features + ['consumed_tonnes'])

    if df_site.empty:
        print(f"Random Forest skipped for {site}: no valid rows.")
        continue

    X = df_site[rf_features]
    y = df_site['consumed_tonnes']

    model = RandomForestRegressor(
        n_estimators=400,
        max_depth=12,
        min_samples_split=4,
        min_samples_leaf=2,
        random_state=42
    )

    model.fit(X, y)
    rf_results[site] = model

    print(f"Random Forest completed for {site}")



Training Random Forest for SITE_001...
Random Forest completed for SITE_001

Training Random Forest for SITE_002...
Random Forest completed for SITE_002

Training Random Forest for SITE_003...
Random Forest completed for SITE_003

Training Random Forest for SITE_004...
Random Forest completed for SITE_004

Training Random Forest for SITE_005...
Random Forest completed for SITE_005

Training Random Forest for SITE_006...
Random Forest completed for SITE_006

Training Random Forest for SITE_007...
Random Forest completed for SITE_007

Training Random Forest for SITE_008...
Random Forest completed for SITE_008

Training Random Forest for SITE_009...
Random Forest completed for SITE_009

Training Random Forest for SITE_010...
Random Forest completed for SITE_010

Training Random Forest for SITE_011...
Random Forest completed for SITE_011

Training Random Forest for SITE_012...
Random Forest completed for SITE_012

Training Random Forest for SITE_013...
Random Forest completed for SITE_013

8 weeks Forcasting pipeline.

In [20]:
def build_future_features(df_site, horizon):
    future_index = pd.date_range(
        start=df_site.index[-1] + pd.Timedelta(days=1),
        periods=horizon,
        freq='D'
    )

    future_df = pd.DataFrame(index=future_index)

    # Static features – repeat last known values
    static_cols = [
        'month','quarter','day_of_week',
        'behavior_aggressive','behavior_chaotic','behavior_conservative'
    ]
    for col in static_cols:
        future_df[col] = df_site[col].iloc[-1]

    # Weather lags – repeat last known values
    future_df['rain_lag_1'] = df_site['rain_lag_1'].iloc[-1]
    future_df['temp_lag_1'] = df_site['temp_lag_1'].iloc[-1]

    # Lag features – use last known values
    future_df['lag_1']  = df_site['lag_1'].iloc[-1]
    future_df['lag_7']  = df_site['lag_7'].iloc[-1]
    future_df['lag_14'] = df_site['lag_14'].iloc[-1]
    future_df['lag_30'] = df_site['lag_30'].iloc[-1]

    # Rolling features – repeat last known values
    future_df['roll_7']  = df_site['roll_7'].iloc[-1]
    future_df['roll_30'] = df_site['roll_30'].iloc[-1]
    future_df['roll_90'] = df_site['roll_90'].iloc[-1]

    # ⭐ FIX: Convert everything to float
    future_df = future_df.astype('float64')

    return future_df


In [21]:
forecast_horizon = 56   # 8 weeks

forecast_results = []

for site in sites:
    print(f"\nForecasting for {site}...")

    df_site = df_features[df_features['site_id'] == site].copy()
    df_site = df_site.sort_values('date').set_index('date').asfreq('D')

    # Build future features
    future_df = build_future_features(df_site, forecast_horizon)

    # -----------------------------
    # SARIMAX Forecast (FIXED)
    # -----------------------------
    sarimax_model = sarimax_results[site]
    sarimax_pred = sarimax_model.get_forecast(
        steps=forecast_horizon,
        exog=future_df[sarimax_features]   # REQUIRED
    )
    sarimax_forecast = sarimax_pred.predicted_mean

    # -----------------------------
    # XGBoost Forecast
    # -----------------------------
    xgb_model = xgb_results[site]
    xgb_forecast = xgb_model.predict(future_df[sarimax_features])

    # -----------------------------
    # Random Forest Forecast
    # -----------------------------
    rf_model = rf_results[site]
    rf_forecast = rf_model.predict(future_df[sarimax_features])

    # -----------------------------
    # Ensemble Forecast (average)
    # -----------------------------
    ensemble_forecast = (
        sarimax_forecast.values +
        xgb_forecast +
        rf_forecast
    ) / 3

    # Store results
    temp = pd.DataFrame({
        'site_id': site,
        'date': future_df.index,
        'sarimax': sarimax_forecast.values,
        'xgboost': xgb_forecast,
        'random_forest': rf_forecast,
        'ensemble': ensemble_forecast
    })

    forecast_results.append(temp)

# Combine all sites
forecast_df = pd.concat(forecast_results)
forecast_df = forecast_df.sort_values(['site_id','date'])



Forecasting for SITE_001...

Forecasting for SITE_002...

Forecasting for SITE_003...

Forecasting for SITE_004...

Forecasting for SITE_005...

Forecasting for SITE_006...

Forecasting for SITE_007...

Forecasting for SITE_008...

Forecasting for SITE_009...

Forecasting for SITE_010...

Forecasting for SITE_011...

Forecasting for SITE_012...

Forecasting for SITE_013...

Forecasting for SITE_014...

Forecasting for SITE_015...

Forecasting for SITE_016...

Forecasting for SITE_017...

Forecasting for SITE_018...

Forecasting for SITE_019...

Forecasting for SITE_020...

Forecasting for SITE_021...

Forecasting for SITE_022...

Forecasting for SITE_023...

Forecasting for SITE_024...

Forecasting for SITE_025...

Forecasting for SITE_026...

Forecasting for SITE_027...

Forecasting for SITE_028...

Forecasting for SITE_029...

Forecasting for SITE_030...


inspect forecast results 

In [22]:
forecast_df.head()
forecast_df.tail()
forecast_df['ensemble'].describe()


count    1680.000000
mean       21.429728
std        15.577417
min        -2.474229
25%         7.069403
50%        19.131071
75%        32.161310
max        53.506644
Name: ensemble, dtype: float64

Insight there is a negative forecast -2.47. Negative values usually come from:

1️⃣ SARIMAX overshooting
SARIMAX can produce negative values if the differencing or residuals swing below zero.

2️⃣ XGBoost extrapolation
XGB can predict slightly negative values if the last few lags were low.

3️⃣ Random Forest averaging
RF can produce small negative values if trained on noisy low‑consumption days.

4️⃣ Ensemble averaging
If one model dips slightly negative, the ensemble can dip too.

This is normal in industrial forecasting.

In [23]:
#Fix: Clamp negative forecasts to zero,Before inventory simulation, a
forecast_df['ensemble'] = forecast_df['ensemble'].clip(lower=0)
forecast_df['sarimax'] = forecast_df['sarimax'].clip(lower=0)
forecast_df['xgboost'] = forecast_df['xgboost'].clip(lower=0)
forecast_df['random_forest'] = forecast_df['random_forest'].clip(lower=0)


Choose the best model per site -s noyt needed since the gola is to deploy. 

SARIMAX training

XGBoost training

Random Forest training

Ensemble logic

Forecasting pipeline